# 03a Export GSOD Daily Data from DuckDB

This notebook reads the local GSOD Philippines DuckDB database and exports selected daily fields to ordinary CSV files.

This notebook is intentionally minimal and does not require plotting libraries.

The exported files are used by the separate GSOD sanity-check notebook.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

In [2]:
GSOD_DB_PATH = Path("/home/jupyter-bbr/notebooks/gsod/gsod_ph.db")

# Put the exported files somewhere accessible to your main analysis environment.
EXPORT_DIR = Path("/home/jupyter-bbr/notebooks/gsod/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 1980
END_YEAR = 2024

GSOD_TABLE = "gsod_daily"

STATION_INVENTORY_PATH = EXPORT_DIR / "gsod_station_inventory_1980_2024.csv"
GSOD_DAILY_EXPORT_PATH = EXPORT_DIR / "gsod_daily_metric_1980_2024.csv"
GSOD_NCR_ADJACENT_EXPORT_PATH = EXPORT_DIR / "gsod_daily_metric_ncr_adjacent_1980_2024.csv"

# Broad NCR-adjacent bounding box for first screening.
NCR_LAT_MIN = 13.8
NCR_LAT_MAX = 15.2
NCR_LON_MIN = 120.4
NCR_LON_MAX = 121.6

In [3]:
if not GSOD_DB_PATH.exists():
    raise FileNotFoundError(f"GSOD database not found: {GSOD_DB_PATH}")

con = duckdb.connect(str(GSOD_DB_PATH), read_only=True)

print(f"Connected to: {GSOD_DB_PATH}")

Connected to: /home/jupyter-bbr/notebooks/gsod/gsod_ph.db


In [4]:
tables = con.execute("SHOW TABLES").df()
display(tables)

,name
0,gsod_daily


In [5]:
schema = con.execute(f"DESCRIBE {GSOD_TABLE}").df()
display(schema)

preview = con.execute(f"""
    SELECT *
    FROM {GSOD_TABLE}
    LIMIT 10
""").df()

display(preview)

,column_name,column_type,null,key,default,extra
0,station_id,VARCHAR,YES,None,None,None
1,station_name,VARCHAR,YES,None,None,None
2,lat,FLOAT,YES,None,None,None
3,lon,FLOAT,YES,None,None,None
4,date,DATE,YES,None,None,None
5,TEMP_C,FLOAT,YES,None,None,None
6,DEWP_C,FLOAT,YES,None,None,None
7,MAX_C,FLOAT,YES,None,None,None
8,MIN_C,FLOAT,YES,None,None,None
9,PRCP_mm,FLOAT,YES,None,None,None


,station_id,station_name,lat,lon,date,TEMP_C,DEWP_C,MAX_C,MIN_C,PRCP_mm,WDSP_ms,MXSPD_ms,GUST_ms,SLP
0,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-01,27.222221,23.888889,31.111111,24.388889,0.0,3.446775,6.173328,NaN,1012.599976
1,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-02,27.277779,24.277779,30.000000,25.000000,0.0,2.417887,5.195884,NaN,1011.900024
2,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-03,25.444445,24.444445,28.888889,23.277779,NaN,1.903443,6.173328,NaN,1011.900024
3,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-04,26.222221,22.222221,28.277779,25.000000,0.0,4.527107,7.716660,NaN,1013.400024
4,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-05,26.000000,20.166666,28.277779,24.388889,0.0,4.578552,7.202216,NaN,1014.799988
5,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-06,26.166666,21.500000,28.277779,24.388889,0.0,4.629996,8.179660,NaN,1014.400024
6,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-07,26.611111,22.888889,29.388889,25.000000,0.0,3.806885,7.202216,NaN,1013.299988
7,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-08,27.444445,23.777779,31.722221,24.388889,NaN,3.652552,7.202216,NaN,1013.500000
8,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-09,27.555555,23.277779,30.000000,25.000000,0.0,3.292442,6.687772,NaN,1013.500000
9,98428041224,SANGLEY POINT,14.5,120.900002,1970-01-10,26.833334,22.222221,30.000000,25.000000,0.0,3.343886,6.173328,NaN,1013.700012


In [6]:
range_check = con.execute(f"""
    SELECT
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS n_rows,
        COUNT(PRCP_mm) AS n_valid_prcp,
        MIN(PRCP_mm) AS min_prcp_mm,
        MAX(PRCP_mm) AS max_prcp_mm
    FROM {GSOD_TABLE}
""").df()

display(range_check)

,min_date,max_date,n_rows,n_valid_prcp,min_prcp_mm,max_prcp_mm
0,1970-01-01,2025-07-01,813411,794147,0.0,500.126007


In [7]:
station_inventory = con.execute(f"""
    SELECT
        station_id,
        station_name,
        MEDIAN(lat) AS latitude,
        MEDIAN(lon) AS longitude,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(*) AS n_rows,
        COUNT(PRCP_mm) AS valid_rainfall_days,
        AVG(PRCP_mm) AS mean_rainfall_mm,
        MAX(PRCP_mm) AS max_rainfall_mm
    FROM {GSOD_TABLE}
    WHERE date BETWEEN '{START_YEAR}-01-01' AND '{END_YEAR}-12-31'
    GROUP BY station_id, station_name
    ORDER BY station_name
""").df()

station_inventory["first_date"] = pd.to_datetime(station_inventory["first_date"])
station_inventory["last_date"] = pd.to_datetime(station_inventory["last_date"])

station_inventory["first_year"] = station_inventory["first_date"].dt.year
station_inventory["last_year"] = station_inventory["last_date"].dt.year

station_inventory["record_years"] = (
    station_inventory["last_year"] - station_inventory["first_year"] + 1
)

station_inventory["inside_ncr_adjacent_bbox"] = (
    station_inventory["latitude"].between(NCR_LAT_MIN, NCR_LAT_MAX)
    & station_inventory["longitude"].between(NCR_LON_MIN, NCR_LON_MAX)
)

station_inventory = station_inventory.sort_values(
    ["inside_ncr_adjacent_bbox", "record_years", "valid_rainfall_days"],
    ascending=[False, False, False],
)

display(station_inventory)

,station_id,station_name,latitude,longitude,first_date,last_date,n_rows,valid_rainfall_days,mean_rainfall_mm,max_rainfall_mm,first_year,last_year,record_years,inside_ncr_adjacent_bbox
63,98430099999,SCIENCE GARDEN,14.650,121.050003,1980-01-01,2024-12-31,15402,15117,5.948224,490.981995,1980,2024,45,True
54,98429099999,NINOY AQUINO INTL,14.509,121.019997,1980-01-01,2024-12-31,16406,14713,2.751411,400.049988,1980,2024,45,True
70,98427099999,TAYABAS,14.017,121.599998,1980-01-01,2024-12-31,14843,14606,7.523189,500.126007,1980,2024,45,True
1,98432099999,AMBULONG,14.083,121.050003,1980-01-01,2024-12-31,12818,12650,4.727372,436.372009,1980,2024,45,True
49,98425099999,MANILA,14.583,120.983002,1981-05-18,2023-11-30,14067,13818,5.339423,427.989990,1981,2023,43,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,98437099999,SAN FRANCISCO,13.367,122.516998,1992-03-12,1993-09-04,341,341,3.350416,97.028000,1992,1993,2,False
41,98830099999,JOLO,6.050,121.000000,1980-03-02,1981-01-31,79,79,1.009570,22.098000,1980,1981,2,False
35,69807499999,GUIUAN,11.033,125.750000,1990-10-01,1990-12-14,46,46,0.000000,0.000000,1990,1990,1,False
67,69705499999,TACLOBAN,11.233,125.032997,1990-07-13,1990-12-02,26,23,8.923131,78.994003,1990,1990,1,False


In [8]:
station_inventory.to_csv(STATION_INVENTORY_PATH, index=False)

print(f"Saved: {STATION_INVENTORY_PATH}")

Saved: /home/jupyter-bbr/notebooks/gsod/exports/gsod_station_inventory_1980_2024.csv


In [9]:
ncr_candidate_stations = station_inventory[
    station_inventory["inside_ncr_adjacent_bbox"]
].copy()

display(
    ncr_candidate_stations[
        [
            "station_id",
            "station_name",
            "latitude",
            "longitude",
            "first_year",
            "last_year",
            "record_years",
            "valid_rainfall_days",
            "mean_rainfall_mm",
            "max_rainfall_mm",
        ]
    ]
)

,station_id,station_name,latitude,longitude,first_year,last_year,record_years,valid_rainfall_days,mean_rainfall_mm,max_rainfall_mm
63,98430099999,SCIENCE GARDEN,14.650,121.050003,1980,2024,45,15117,5.948224,490.981995
54,98429099999,NINOY AQUINO INTL,14.509,121.019997,1980,2024,45,14713,2.751411,400.049988
70,98427099999,TAYABAS,14.017,121.599998,1980,2024,45,14606,7.523189,500.126007
1,98432099999,AMBULONG,14.083,121.050003,1980,2024,45,12650,4.727372,436.372009
49,98425099999,MANILA,14.583,120.983002,1981,2023,43,13818,5.339423,427.989990
62,98428099999,SANGLEY POINT AB,14.495,120.903999,2000,2024,25,8591,6.117051,354.075989
69,98433099999,TANAY,14.583,121.366997,2001,2024,24,7416,7.795841,360.171997
61,98428041224,SANGLEY POINT,14.500,120.900002,1980,1999,20,6255,3.036427,329.946014
19,69176499999,CLARK INTL,15.183,120.567001,1985,2003,19,192,0.760677,19.049999
73,69185499999,VILLAMOR,14.517,121.016998,1993,2000,8,14,0.381000,3.302000


In [10]:
gsod_daily_export = con.execute(f"""
    SELECT
        CAST(station_id AS VARCHAR) AS STATION_ID,
        CAST(station_name AS VARCHAR) AS STATION_NAME,
        CAST(lat AS DOUBLE) AS LATITUDE,
        CAST(lon AS DOUBLE) AS LONGITUDE,
        CAST(date AS DATE) AS DATE,
        EXTRACT(year FROM date) AS YEAR,
        EXTRACT(month FROM date) AS MONTH,
        EXTRACT(day FROM date) AS DAY,
        CAST(PRCP_mm AS DOUBLE) AS RAINFALL
    FROM {GSOD_TABLE}
    WHERE date BETWEEN '{START_YEAR}-01-01' AND '{END_YEAR}-12-31'
    ORDER BY STATION_ID, DATE
""").df()

display(gsod_daily_export.head())
print(gsod_daily_export.shape)

,STATION_ID,STATION_NAME,LATITUDE,LONGITUDE,DATE,YEAR,MONTH,DAY,RAINFALL
0,69176499999,CLARK INTL,15.183,120.567001,1985-03-12,1985,3,12,0.0
1,69176499999,CLARK INTL,15.183,120.567001,1986-04-18,1986,4,18,NaN
2,69176499999,CLARK INTL,15.183,120.567001,1986-04-19,1986,4,19,NaN
3,69176499999,CLARK INTL,15.183,120.567001,1986-04-22,1986,4,22,NaN
4,69176499999,CLARK INTL,15.183,120.567001,1986-04-23,1986,4,23,NaN


(724377, 9)


In [11]:
gsod_daily_export.to_csv(GSOD_DAILY_EXPORT_PATH, index=False)

print(f"Saved: {GSOD_DAILY_EXPORT_PATH}")

Saved: /home/jupyter-bbr/notebooks/gsod/exports/gsod_daily_metric_1980_2024.csv


In [12]:
gsod_ncr_adjacent_export = con.execute(f"""
    SELECT
        CAST(station_id AS VARCHAR) AS STATION_ID,
        CAST(station_name AS VARCHAR) AS STATION_NAME,
        CAST(lat AS DOUBLE) AS LATITUDE,
        CAST(lon AS DOUBLE) AS LONGITUDE,
        CAST(date AS DATE) AS DATE,
        EXTRACT(year FROM date) AS YEAR,
        EXTRACT(month FROM date) AS MONTH,
        EXTRACT(day FROM date) AS DAY,
        CAST(PRCP_mm AS DOUBLE) AS RAINFALL
    FROM {GSOD_TABLE}
    WHERE date BETWEEN '{START_YEAR}-01-01' AND '{END_YEAR}-12-31'
      AND lat BETWEEN {NCR_LAT_MIN} AND {NCR_LAT_MAX}
      AND lon BETWEEN {NCR_LON_MIN} AND {NCR_LON_MAX}
    ORDER BY STATION_ID, DATE
""").df()

display(gsod_ncr_adjacent_export.head())
print(gsod_ncr_adjacent_export.shape)

,STATION_ID,STATION_NAME,LATITUDE,LONGITUDE,DATE,YEAR,MONTH,DAY,RAINFALL
0,69176499999,CLARK INTL,15.183,120.567001,1985-03-12,1985,3,12,0.0
1,69176499999,CLARK INTL,15.183,120.567001,1986-04-18,1986,4,18,NaN
2,69176499999,CLARK INTL,15.183,120.567001,1986-04-19,1986,4,19,NaN
3,69176499999,CLARK INTL,15.183,120.567001,1986-04-22,1986,4,22,NaN
4,69176499999,CLARK INTL,15.183,120.567001,1986-04-23,1986,4,23,NaN


(97347, 9)


In [13]:
gsod_ncr_adjacent_export.to_csv(GSOD_NCR_ADJACENT_EXPORT_PATH, index=False)

print(f"Saved: {GSOD_NCR_ADJACENT_EXPORT_PATH}")

Saved: /home/jupyter-bbr/notebooks/gsod/exports/gsod_daily_metric_ncr_adjacent_1980_2024.csv


In [14]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
